     Chapter 11. Fine-Tuning Representation Models for Classification

                  Supervised Classification
                  Fine-Tuning a Pretrained BERT Model

In [ ]:
# Install specific versions of Hugging Face libraries
# to keep them compatible with each other.

!pip install \
transformers==4.46.3 \       # Main library for pretrained models like BERT
datasets==3.2.0 \            # Used to load and process datasets
huggingface_hub==0.26.2 \    # Used to download models and datasets from Hugging Face
tokenizers==0.20.3 \         # Used for fast text tokenization
accelerate==1.1.1            # Helps with efficient training on CPU/GPU

In [ ]:
# Import the load_dataset function from the Hugging Face datasets library
from datasets import load_dataset

# Load the Rotten Tomatoes dataset
tomatoes = load_dataset("rotten_tomatoes")

# Separate the dataset into training and test sets
# Get the training part of the dataset
train_data = tomatoes["train"]

# Get the testing part of the dataset
test_data = tomatoes["test"]

In [ ]:
# Import AutoTokenizer to convert text into tokens and token IDs
# Import AutoModelForSequenceClassification to load BERT
    # with a classification layer for predicting classes

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Select the pretrained BERT model
model_id = "bert-base-cased"


# Load the pretrained BERT model
# AutoModelForSequenceClassification adds a classification head
    # on top of BERT so it can classify the input text
# num_labels=2 means we have two possible classes:
# 0 = Negative
# 1 = Positive
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2
)

# Load the tokenizer suitable for this BERT model
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# DataCollatorWithPadding =  It automatically adds padding to the input sequences.
from transformers import DataCollatorWithPadding

# Create a data collator that dynamically adds padding
# to make sequences in each batch the same length
data_collator = DataCollatorWithPadding(

       # Use the same tokenizer that we loaded for BERT.
    # The tokenizer knows which padding token to use.
    tokenizer=tokenizer
)

In [ ]:
def preprocess_function(examples):
    # Take the text from the dataset
    # Tokenize the text
    # Truncate sequences that are too long
    return tokenizer(
        examples["text"],
        truncation=True
    )

In [ ]:
# Apply the preprocessing function to the training dataset
# batched=True means multiple examples are processed together
tokenized_train = train_data.map(
    preprocess_function,
    batched=True
)

In [ ]:
# Apply the same preprocessing to the test dataset
tokenized_test = test_data.map(
    preprocess_function,
    batched=True
)

In [ ]:
# upgrade the required datasets and scikit-learn versions
!pip install -U "datasets==2.18.0" "scikit-learn==1.3.2"

In [ ]:
!pip uninstall -y datasets
!pip install datasets==2.18.0

In [ ]:
# Import NumPy for numerical operations
import numpy as np

# Import the function used to load evaluation metrics
from datasets import load_metric

 # Function to calculate the F1 score
def compute_metrics(eval_pred):

    # Get the model's raw scores and the correct labels
    logits, labels = eval_pred

    # Select the class with the highest score
    predictions = np.argmax(logits, axis=-1)

    # Load the F1 metric
    load_f1 = load_metric("f1")

    # Compare predictions with the correct labels
    # predictions = what the model predicted
    # references = the correct answers from the dataset

    f1 = load_f1.compute(
        predictions=predictions,
        references=labels
    )["f1"]       # Get only the F1 score from the result

    # Return the F1 score
    return {"f1": f1}

In [ ]:
from transformers import TrainingArguments

# Define how the model should be trained
training_args = TrainingArguments(
    "model",                         # Folder to save the model/checkpoints
    learning_rate=2e-5,              # Size of weight updates
    per_device_train_batch_size=16, # 16 examples per training batch
    per_device_eval_batch_size=16,  # 16 examples per evaluation batch
    num_train_epochs=1,              # Go through training data once
    weight_decay=0.01,               # Helps reduce overfitting
    save_strategy="epoch",           # Save after each epoch
    report_to="none"                 # Don't report to external services
)

In [ ]:
# Import Trainer to handle the training and evaluation process
from transformers import Trainer

# Create the Trainer and give it everything needed for training
trainer = Trainer(

    # The BERT model that will be trained
    model=model,

     # Training settings such as learning rate, batch size, and epochs
    args=training_args,

    # Tokenized training data used to train the model
    train_dataset=tokenized_train,

    # Tokenized test data used to evaluate the model
    eval_dataset=tokenized_test,

     # BERT tokenizer used to convert text into tokens
    tokenizer=tokenizer,

     # Dynamically pads the sequences in each batch
    data_collator=data_collator,

     # Function used to calculate the F1 score during evaluation
    compute_metrics=compute_metrics,
)

In [ ]:
# Start the actual training/fine-tuning process
trainer.train()

In [ ]:
# Evaluate the trained model on the test dataset
trainer.evaluate()

                             Freezing Layers

In [ ]:
# Import the tokenizer and the model class for sequence classification
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Select the pretrained BERT model
model_id = "bert-base-cased"

# Load BERT and add a classification head
# There are 2 classes: Positive and Negative
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2
)

# Load the tokenizer suitable for this BERT model
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# Print the names of all layers and parameters in the BERT model
for name, param in model.named_parameters():
  print(name)

In [ ]:
# Go through every parameter in the BERT model
for name, param in model.named_parameters():

# Check whether the parameter belongs to the classification head
 if name.startswith("classifier"):

  # Allow the classification head to be updated during training
  param.requires_grad = True

  # If the parameter is not part of the classification head
else:
   # Freeze that parameter so its values do not change during training
  param.requires_grad = False

In [ ]:
from transformers import TrainingArguments, Trainer
# Trainer which executes the training process
trainer = Trainer(
model=model,
args=training_args,
train_dataset=tokenized_train,
eval_dataset=tokenized_test,
tokenizer=tokenizer,
data_collator=data_collator,
compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
trainer.evaluate()

               Freezing first 10 encoder blocks of BERT model

In [ ]:
# Import the tokenizer and the model class for sequence classification
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Select the pretrained BERT model
model_id = "bert-base-cased"

# Load BERT and add a classification head
# There are 2 classes: Positive and Negative
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=2
)

# Load the tokenizer suitable for this BERT model
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
# Go through all model parameters with their index and name
for index, (name, param) in enumerate(model.named_parameters()):

  # Freeze all parameters before index 165
  # This means the earlier BERT layers will not be updated during training

  if index < 165:

    # Stop these parameters from learning/updating
    param.requires_grad = False

In [ ]:
# Trainer which executes the training process
trainer = Trainer(
model=model,
args=training_args,
train_dataset=tokenized_train,
eval_dataset=tokenized_test,
tokenizer=tokenizer,
data_collator=data_collator,
compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

                    Few-Shot Classification
                    SetFit: Efficient Fine-Tuning with Few Training Examples

In [ ]:
 # Install the SetFit library to the latest compatible version
!pip install -U setfit

In [ ]:
Using the previously loaded dataset

In [ ]:
# Import sample_dataset from SetFit
# It is used to select a small number of examples from each class
from setfit import sample_dataset


# We simulate a few-shot setting by sampling 16 examples per class
sampled_train_data = sample_dataset(tomatoes["train"],
num_samples=16)

In [ ]:
# SetFitModel is used to build and train SetFit models
from setfit import SetFitModel

# Load a pretrained SentenceTransformer model
# "all-mpnet-base-v2" converts sentences into meaningful numerical embeddings
# SetFit will use these embeddings to perform classification
model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")

In [ ]:
# Import SetFit's TrainingArguments class
   # We rename it to SetFitTrainingArguments to make it clear that
   # these arguments are specifically for SetFit
from setfit import TrainingArguments as SetFitTrainingArguments

# SetFitTrainer is responsible for training and evaluating the SetFit model
from setfit import Trainer as SetFitTrainer

# Define training arguments
args = SetFitTrainingArguments(
num_epochs=3, # The number of epochs to use for contrastive learning
num_iterations=20  # The number of text pairs to generate
)

# Set eval_strategy using the value stored in evaluation_strategy
# This helps maintain compatibility between different SetFit versions
args.eval_strategy = args.evaluation_strategy

In [ ]:
!pip install -q \
transformers==4.41.2 \
sentence-transformers==3.0.1 \
setfit==1.0.3 \
huggingface_hub==0.23.5 \
accelerate==0.31.0 \
peft==0.11.1

In [ ]:
# Create the SetFit trainer
trainer = SetFitTrainer(

    # The pretrained SetFit mode
    model=model,

    # Training settings defined earlier
    args=args,

    # Small few-shot training dataset
train_dataset=sampled_train_data,

     # Test dataset used for evaluation
eval_dataset=test_data,

    # Use F1 score to measure performance

metric="f1"
)

# Training loop
trainer.train()

In [ ]:
# Evaluate the model on our test data
trainer.evaluate()

            Continued Pretraining with Masked Language Modeling

In [ ]:
# Import the tokenizer and MLM model classes
from transformers import AutoTokenizer, AutoModelForMaskedLM

# MLM means the model learns to predict masked/missing words in a sentence
model = AutoModelForMaskedLM.from_pretrained("bert-base-cased")

# The tokenizer converts text into tokens that BERT can understand
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
# Define a function to tokenize the input text
def preprocess_function(examples):

  # Take the text and convert it into tokens
    # truncation=True cuts the text if it is too long
  return tokenizer(examples["text"], truncation=True
)

# Apply the preprocessing function to the training dataset
tokenized_train = train_data.map(preprocess_function,

  # batched=True processes multiple examples at once
batched=True
)

# Remove the original sentiment labels
# MLM does not use the positive/negative labels
tokenized_train = tokenized_train.remove_columns("label")

tokenized_test = test_data.map(preprocess_function, batched=True
)

# Remove the original sentiment labels
# MLM does not use the positive/negative labels
tokenized_test = tokenized_test.remove_columns("label")

In [ ]:
# Import the data collator used for Masked Language Modeling
from transformers import DataCollatorForLanguageModeling


# Create a data collator for masking tokens
data_collator = DataCollatorForLanguageModeling(

  # Use the BERT tokenizer to tokenize and mask the text
tokenizer=tokenizer,

   # Enable Masked Language Modeling
  # This tells the collator to randomly hide tokens
mlm=True,

   # Mask 15% of the tokens in each input
mlm_probability=0.15
)

# Training arguments for parameter tuning
training_args = TrainingArguments(
"model",
learning_rate=2e-5,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
num_train_epochs=10,
weight_decay=0.01,
save_strategy="epoch",
report_to="none"
)


# Create the Trainer to manage the MLM training process
trainer = Trainer(
model=model,
args=training_args,
train_dataset=tokenized_train,
eval_dataset=tokenized_test,
tokenizer=tokenizer,
data_collator=data_collator
)


In [ ]:
# Save pre-trained tokenizer
tokenizer.save_pretrained("mlm")

# Train model
trainer.train()

# Save updated model
model.save_pretrained("mlm")

In [ ]:
# Import the pipeline function from Transformers
from transformers import pipeline

# Create a fill-mask pipeline
# This pipeline is used to predict a missing word represented by [MASK]
mask_filler = pipeline("fill-mask", model="bert-base-cased")

# Give BERT a sentence containing [MASK]
# BERT will predict possible words that can replace [MASK]
preds = mask_filler("What a horrible [MASK]!")

# Go through each prediction returned by the model
for pred in preds:

  # Print the complete sentence after replacing [MASK] with the predicted word
  print(f">>> {pred["sequence"]}")

In [ ]:
# Load the BERT model that we continued pretraining and saved in the "mlm" folder
# The "fill-mask" pipeline is used to predict a missing [MASK] token
mask_filler = pipeline("fill-mask", model="mlm")

# Give the model a sentence containing a [MASK] token
# The model predicts possible words that can replace [MASK]
preds = mask_filler("What a horrible [MASK]!")

# Go through each prediction returned by the model
for pred in preds:

   # Print the complete sentence after replacing [MASK] with the predicted word
  print(f">>> {pred["sequence"]}")

In [ ]:
# Import the model class used for text classification
from transformers import AutoModelForSequenceClassification

# Load the BERT model that we previously continued pretraining using MLM
# "mlm" is the folder containing our updated BERT model
# Add a classification head with 2 output classes: Positive and Negative

model = AutoModelForSequenceClassification.from_pretrained("mlm",
num_labels=2)

# Load the tokenizer that belongs to our continuously pretrained model
# We use the same tokenizer that was saved with the "mlm" model
tokenizer = AutoTokenizer.from_pretrained("mlm")


                      Named-Entity Recognition

In [ ]:
# Remove the currently installed versions of these Hugging Face libraries
!pip uninstall -y datasets huggingface_hub transformers

# Install specific versions to keep the libraries compatible with each other
!pip install datasets==3.6.0 transformers==4.46.3 huggingface_hub==0.26.2

In [ ]:
# Load the CoNLL-2003 dataset, which is commonly used for Named-Entity Recognition (NER)
# trust_remote_code=True allows the dataset loading script to run
dataset = load_dataset("conll2003", trust_remote_code=True)


In [ ]:
# Select the 849th example from the training dataset
# Index 848 means we are selecting the example at position 848
example = dataset["train"][848]

example

In [ ]:
# Create a dictionary that maps each NER label to a unique number
label2id = {

    # O means the token is not a named entity
    "O": 0,

    # B-PER means the beginning of a person's name
    "B-PER": 1,

    # I-PER means inside/continuation of a person's name
    "I-PER": 2,

    # B-ORG means the beginning of an organization name
    "B-ORG": 3,

    # I-ORG means inside/continuation of an organization name
    "I-ORG": 4,

    # B-LOC means the beginning of a location name
    "B-LOC": 5,

    # I-LOC means inside/continuation of a location name
    "I-LOC": 6,

    # B-MISC means the beginning of a miscellaneous entity
    "B-MISC": 7,

    # I-MISC means inside/continuation of a miscellaneous entity
    "I-MISC": 8
}

In [ ]:
# Create a reverse dictionary that maps each ID back to its NER label
id2label = {index: label
for label, index in label2id.items()}

# Display the original label-to-ID dictionary
label2id

In [ ]:
# Import the model class used for token-level classification
from transformers import AutoModelForTokenClassification

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# Load the pretrained BERT model with a token-classification head
model = AutoModelForTokenClassification.from_pretrained(

  # Use the pretrained BERT base cased model
"bert-base-cased",

   # Tell the model how many different NER labels it needs to predict
num_labels=len(id2label),

   # Tell the model which ID corresponds to which label
id2label=id2label,

  # Tell the model which label corresponds to which ID
label2id=label2id
)

In [ ]:
# Define a function to tokenize the words and align their NER labels
def align_labels(examples):

    # Tokenize the words into BERT tokens/subwords
    # examples["tokens"] contains the words from the CoNLL-2003 dataset
    token_ids = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    # Get the original NER labels for each word
    # Example: [1, 2, 0, 0, 3]
    labels = examples["ner_tags"]

    # Create an empty list to store the new labels
    # These labels will match the BERT tokens
    updated_labels = []

    # Go through each sentence in the batch
    # index = sentence number
    # label = NER labels belonging to that sentence
    for index, label in enumerate(labels):

        # Get the original word number for every BERT token
        # This tells us which word each token came from
        word_ids = token_ids.word_ids(batch_index=index)

        # Keep track of the previous word number
        # We start with None because there is no previous word
        previous_word_idx = None

        # Create an empty list to store labels for this sentence
        label_ids = []

        # Go through every token in the sentence
        for word_idx in word_ids:

            # Check whether this token belongs to a new word
            if word_idx != previous_word_idx:

                # Remember the current word number
                previous_word_idx = word_idx

                # If word_idx is None, this is a special token such as [CLS] or [SEP]
                # Otherwise, get the original NER label of that word
                updated_label = -100 if word_idx is None else label[word_idx]

                # Add the label to the list
                label_ids.append(updated_label)

            # Handle special tokens such as [CLS] and [SEP]
            elif word_idx is None:

                # Use -100 so the model ignores these tokens during loss calculation
                label_ids.append(-100)

            # This token is another subtoken of the same word
            else:

                # Get the original NER label of the word
                updated_label = label[word_idx]

                # If the label is B-XXX, change it to I-XXX
                # This is needed because this token is inside the word,
                # not the beginning of the entity
                if updated_label % 2 == 1:
                    updated_label += 1

                # Add the updated label
                label_ids.append(updated_label)

        # Store the labels for this sentence
        updated_labels.append(label_ids)

    # Add the new token-level labels to the tokenized data
    token_ids["labels"] = updated_labels

    # Return the tokenized data with the aligned labels
    return token_ids


# Apply align_labels to the entire dataset
# batched=True means multiple sentences are processed together
tokenized = dataset.map(
    align_labels,
    batched=True
)

In [ ]:
# Print the original NER labels from the dataset
print(f"Original: {example['ner_tags']}")

# Print the updated labels after tokenization and label alignment
print(f"Updated: {tokenized['train'][848]['labels']}")

In [ ]:
!pip install -U evaluate seqeval

In [ ]:
import evaluate

# Load the evaluation library for Named Entity Recognition
seqeval = evaluate.load("seqeval")


def compute_metrics(eval_pred):

    # Separate model predictions and true labels
    logits, labels = eval_pred

    # Convert logits into predicted label IDs
    predictions = np.argmax(logits, axis=2)

    # Lists to store predicted and actual labels
    true_predictions = []
    true_labels = []

    # Loop through each sentence
    for prediction, label in zip(predictions, labels):

        # Loop through each token in the sentence
        for token_prediction, token_label in zip(prediction, label):

            # Ignore special tokens ([CLS], [SEP], padding)
            if token_label != -100:

                # Convert predicted ID to label name
                true_predictions.append(
                    [id2label[token_prediction]]
                )

                # Convert actual ID to label name
                true_labels.append(
                    [id2label[token_label]]
                )

    # Calculate evaluation metrics
    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    # Return F1 score
    return {
        "f1": results["overall_f1"]
    }

              Fine-Tuning for Named-Entity Recognition

In [ ]:
from transformers import DataCollatorForTokenClassification

# Create a data collator for NER
# It dynamically pads the input tokens and their corresponding labels
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
# Training arguments for parameter tuning
training_args = TrainingArguments(
"model",
learning_rate=2e-5,
per_device_train_batch_size=16,
per_device_eval_batch_size=16,
num_train_epochs=1,
weight_decay=0.01,
save_strategy="epoch",
report_to="none"
)

In [ ]:
# Initialize Trainer
trainer = Trainer(
model=model,
args=training_args,
train_dataset=tokenized["train"],
eval_dataset=tokenized["test"],
tokenizer=tokenizer,
data_collator=data_collator,
compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
import numpy as np
# Evaluate the model on our test data
trainer.evaluate()

In [ ]:
# Import the pipeline function from Transformers
from transformers import pipeline


# Save the fine-tuned NER model
# "ner_model" is the folder where the trained model will be saved
trainer.save_model("ner_model")


# Create a pipeline for token classification
# Token classification is used for tasks like NER
token_classifier = pipeline(
    "token-classification",

    # Load our fine-tuned NER model from the saved folder
    model="ner_model",
)


# Give a new sentence to the trained model
# The model will identify named entities in this sentence
token_classifier("My name is Maarten.")